# IDOM ハンズオン Day 2

## 内容
1. **Cortex Search Service** — 商談活動履歴のセマンティック検索
2. **Cortex Analyst** — 構造化データに対する意味づけ層を定義
3. **Cortex Agent** — 構造化データ × 非構造化データを統合した AI エージェント

> **前提**: Day 1 で `DT_SALES_PERFORMANCE` Dynamic Table、`V_ACTIVITY_LOGS` View が作成済みであること

In [ ]:
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

---
## 1. Cortex Search Service の構築

Cortex Search は全文検索＋セマンティック検索を提供するマネージドサービスです。
`V_ACTIVITY_LOGS` ビュー（VARIANT展開済み）を CONTRACTS / STORES と結合し、
検索テキストを動的に生成して Cortex Search に投入します。

### Step 1: V_ACTIVITY_LOGS の確認

Day 1 で作成した `V_ACTIVITY_LOGS` ビューを確認します。
ACTIVITY_LOGS テーブル（VARIANT/JSON）を展開したビューで、
Cortex Search のソースとして直接利用します。

In [ ]:
USE SCHEMA IDOM_HANDSON.DATA_MART;

SELECT * FROM IDOM_HANDSON.RAW_DATA.V_ACTIVITY_LOGS LIMIT 5;

In [ ]:
SELECT COUNT(*) AS LOG_COUNT FROM IDOM_HANDSON.RAW_DATA.V_ACTIVITY_LOGS;

### Step 2: Cortex Search Service の作成

`V_ACTIVITY_LOGS` を `CONTRACTS` / `STORES` と JOIN し、
店舗名・地域・車種名・商談ステータスなどのメタデータを補完します。
`SEARCH_TEXT` は `CONCAT` で動的に生成します。

専用テーブルへの INSERT は不要で、View から直接 Cortex Search を構築できます。

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE IDOM_HANDSON.DATA_MART.ACTIVITY_SEARCH
    ON SEARCH_TEXT
    ATTRIBUTES NEGOTIATION_ID, STORE_NAME, REGION, CUSTOMER_NAME, CAR_NAME,
               CONTRACT_STATUS, ASSIGNED_USER_NAME, ACTIVITY_TYPE, SUBJECT
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 hour'
AS (
    SELECT
        A.ACTIVITY_ID,
        A.NEGOTIATION_ID,
        A.STORE_ID,
        S.STORE_NAME,
        S.REGION,
        A.CUSTOMER_NAME,
        C.CAR_NAME,
        C.CONTRACT_STATUS,
        A.ASSIGNED_USER_NAME,
        A.ACTIVITY_TYPE,
        A.ACTIVITY_DATE,
        A.SUBJECT,
        CONCAT(
            '【商談ID】', A.NEGOTIATION_ID,
            ' 【顧客名】', A.CUSTOMER_NAME,
            ' 【車両】', COALESCE(C.CAR_NAME, '不明'),
            ' 【店舗】', COALESCE(S.STORE_NAME, '不明'), '(', COALESCE(S.REGION, ''), ')',
            ' 【担当】', A.ASSIGNED_USER_NAME,
            ' 【商談ステータス】', COALESCE(C.CONTRACT_STATUS, '不明'),
            ' 【活動種別】', A.ACTIVITY_TYPE,
            ' 【日時】', TO_VARCHAR(A.ACTIVITY_DATE, 'YYYY-MM-DD HH24:MI'),
            ' 【件名】', A.SUBJECT,
            ' 【内容】', A.BODY
        ) AS SEARCH_TEXT
    FROM IDOM_HANDSON.RAW_DATA.V_ACTIVITY_LOGS A
    LEFT JOIN IDOM_HANDSON.RAW_DATA.CONTRACTS C
        ON A.NEGOTIATION_ID = C.NEGOTIATION_ID
    LEFT JOIN IDOM_HANDSON.RAW_DATA.STORES S
        ON A.STORE_ID = S.STORE_ID
);

### Step 3: テスト検索

Cortex Search Service をSQL関数で呼び出してみましょう。

In [ ]:
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'IDOM_HANDSON.DATA_MART.ACTIVITY_SEARCH',
        '{
            "query": "成約した商談の経緯を教えてください",
            "columns": ["ACTIVITY_ID", "CUSTOMER_NAME", "SUBJECT", "CONTRACT_STATUS"],
            "limit": 5
        }'
    )
) AS SEARCH_RESULT;

In [ ]:
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'IDOM_HANDSON.DATA_MART.ACTIVITY_SEARCH',
        '{
            "query": "価格交渉や値引き交渉の記録",
            "columns": ["ACTIVITY_ID", "CUSTOMER_NAME", "CAR_NAME", "SUBJECT"],
            "limit": 5
        }'
    )
) AS SEARCH_RESULT;

In [ ]:
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'IDOM_HANDSON.DATA_MART.ACTIVITY_SEARCH',
        '{
            "query": "上田浩二さんとのやりとり",
            "columns": ["ACTIVITY_ID", "SUBJECT", "ACTIVITY_TYPE", "CONTRACT_STATUS"],
            "limit": 5
        }'
    )
) AS SEARCH_RESULT;

---
## 2. Cortex Analyst（Semantic View）

Cortex Analyst は構造化データに対する「意味づけ層」を定義し、
自然言語 → SQL 変換を可能にするサービスです。
Dynamic Table `DT_SALES_PERFORMANCE` に対して Semantic View を作成します。

Semantic View では以下を定義:
- **facts**: 数値カラム（売上額、契約件数、保険料など）
- **dimensions**: 分析軸（店舗、地域、月など）
- **metrics**: 集計指標（月間売上合計、平均保険付帯率など）

### Step 1: Semantic View の作成

Semantic View は Dynamic Table の上に「意味付け」を行うレイヤーです。
カラムを facts（数値）、dimensions（分析軸）、metrics（集計指標）に分類し、
Cortex Analyst が自然言語から正しいSQLを生成できるようにします。

In [ ]:
USE SCHEMA IDOM_HANDSON.DATA_MART;

CREATE OR REPLACE SEMANTIC VIEW IDOM_HANDSON.DATA_MART.SV_SALES_PERFORMANCE
    tables (
        SALES as IDOM_HANDSON.DATA_MART.DT_SALES_PERFORMANCE
            primary key (STORE_ID, CONTRACT_MONTH)
            comment = '店舗別月別の売上実績データ'
    )
    facts (
        SALES.CONTRACT_COUNT_FACT as CONTRACT_COUNT comment = '成約件数',
        SALES.TOTAL_SALES_FACT as TOTAL_SALES comment = '売上合計金額（円）',
        SALES.AVG_CONTRACT_PRICE_FACT as AVG_CONTRACT_PRICE comment = '平均契約単価（円）',
        SALES.TOTAL_LOAN_AMOUNT_FACT as TOTAL_LOAN_AMOUNT comment = 'ローン総額',
        SALES.DELIVERED_COUNT_FACT as DELIVERED_COUNT comment = '納車済み件数',
        SALES.PROCESSING_COUNT_FACT as PROCESSING_COUNT comment = '手続中件数',
        SALES.CANCELLED_COUNT_FACT as CANCELLED_COUNT comment = 'キャンセル件数',
        SALES.INSURANCE_COUNT_FACT as INSURANCE_COUNT comment = '保険付帯件数',
        SALES.TOTAL_INSURANCE_PREMIUM_FACT as TOTAL_INSURANCE_PREMIUM comment = '保険料合計',
        SALES.INSURANCE_ATTACH_RATE_FACT as INSURANCE_ATTACH_RATE comment = '保険付帯率',
        SALES.TARGET_MONTHLY_SALES_FACT as TARGET_MONTHLY_SALES comment = '月間売上目標',
        SALES.TARGET_MONTHLY_CONTRACTS_FACT as TARGET_MONTHLY_CONTRACTS comment = '月間成約目標件数'
    )
    dimensions (
        SALES.STORE_ID_DIM as STORE_ID comment = '店舗ID',
        SALES.STORE_NAME_DIM as STORE_NAME with synonyms = ('店舗') comment = '店舗名',
        SALES.REGION_DIM as REGION with synonyms = ('エリア') comment = '地域',
        SALES.PREFECTURE_DIM as PREFECTURE comment = '都道府県',
        SALES.STORE_TYPE_DIM as STORE_TYPE comment = '店舗タイプ',
        SALES.CONTRACT_MONTH_DIM as CONTRACT_MONTH with synonyms = ('月', '年月') comment = '契約月'
    )
    metrics (
        SALES.TOTAL_MONTHLY_SALES as SUM(total_sales_fact) comment = '月間売上合計',
        SALES.TOTAL_MONTHLY_CONTRACTS as SUM(contract_count_fact) comment = '月間成約数合計',
        SALES.AVG_INSURANCE_RATE as AVG(insurance_attach_rate_fact) comment = '平均保険付帯率'
    );

### Step 2: Semantic View の確認

作成した Semantic View の定義を確認します。

In [ ]:
DESCRIBE SEMANTIC VIEW IDOM_HANDSON.DATA_MART.SV_SALES_PERFORMANCE;

---
## 3. Cortex Agent の構築

Cortex Search（非構造化データ検索）と Semantic View（構造化データ分析）を
統合した AI エージェントを作成します。

構成:
- **Tool 1** `sales_performance_analyst`: 売上実績の数値分析（Semantic View経由）
- **Tool 2** `activity_search`: 商談活動履歴の全文検索（Cortex Search経由）

### Step 1: Cortex Agent の作成

2つのツールを組み合わせた AI エージェントを作成:
1. **sales_performance_analyst**: 売上の数値分析（Semantic View経由）
2. **activity_search**: 商談活動履歴の全文検索（Cortex Search経由）

In [ ]:
USE SCHEMA IDOM_HANDSON.AGENTS;

CREATE OR REPLACE AGENT IDOM_HANDSON.AGENTS.IDOM_SALES_AGENT
FROM SPECIFICATION
$$
models:
  orchestration: "auto"
orchestration:
  budget:
    seconds: 900
    tokens: 400000
instructions:
  response: "日本語で回答してください。数値データはテーブル形式で見やすく表示し、活動履歴の引用時は具体的な日時と内容を含めてください。"
  orchestration: >
    あなたはIDOMガリバーの営業分析アシスタントです。
    構造化データ（売上実績）と非構造化データ（商談活動履歴）の両方を活用して、
    営業チームの意思決定を支援します。

    質問の種類に応じて適切なツールを使い分けてください：
    - 売上実績・契約件数・保険付帯率などの数値分析 → sales_performance_analyst
    - 顧客との商談経緯・対応履歴・やり取りの詳細 → activity_search

    特に activity_search は以下のような質問に使ってください：
    - 「○○さんの商談の経緯を教えて」
    - 「価格交渉はどのように進んだ？」
    - 「なぜ失注したのか詳しく教えて」
    - 「LINEでのやり取りの内容は？」
    - 「担当者の対応内容を確認したい」

    数値データと活動履歴を組み合わせて回答することで、より深いインサイトを提供してください。
tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "sales_performance_analyst"
      description: >
        店舗別・月別の売上実績、契約件数、平均契約単価、保険付帯率、
        目標達成率などの構造化データを分析します。
        売上トレンド、地域別実績比較、保険クロスセル分析に使用してください。
  - tool_spec:
      type: "cortex_search"
      name: "activity_search"
      description: >
        商談の活動履歴（電話メモ、メール内容、来店対応記録、LINE対話、
        社内メモ、契約手続き記録）を全文検索します。
        顧客名、車種名、店舗名、担当者名、商談ステータスで絞り込めます。
tool_resources:
  sales_performance_analyst:
    execution_environment:
      query_timeout: 299
      type: "warehouse"
      warehouse: ""
    semantic_view: "IDOM_HANDSON.DATA_MART.SV_SALES_PERFORMANCE"
  activity_search:
    execution_environment:
      query_timeout: 299
      type: "warehouse"
      warehouse: ""
    search_service: "IDOM_HANDSON.DATA_MART.ACTIVITY_SEARCH"
$$;

### Step 2: Agent をテスト

Snowsight の Agents ページからテストできます:

左メニュー → **AI & ML** → **Agents** → **IDOM_SALES_AGENT**

サンプル質問:
- 「売上が一番高い店舗はどこ？」
- 「地域別の契約件数を教えて」
- 「上田浩二さんの商談はどのように進みましたか？」
- 「価格交渉で失注した案件の詳細を教えて」
- 「東京本店の保険付帯率は？」

---
## Day 2 まとめ

| 項目 | 学んだこと |
|------|----------|
| Cortex Search | V_ACTIVITY_LOGS から直接セマンティック検索サービスを構築 |
| Cortex Analyst | Semantic View で構造化データに意味づけ（facts / dimensions / metrics） |
| Cortex Agent | 構造化 × 非構造化データを統合した AI エージェント |

構造化データ（売上実績）と非構造化データ（商談活動履歴）を
統合した AI 営業アシスタントが完成しました！